In [ ]:
import os
import pandas as pd
from scipy.stats import chi2_contingency
from statsmodels.stats.contingency_tables import Table
import math
from collections import defaultdict
from typing import Tuple

In [2]:
sp18_ext_path = "../resources/developed/sp18/external_effect"

res_outdir = "../out/external_effect"

outcsv_args = {
    "sep": "\t",
    "index": True,
    "header": True
}

In [7]:
# Compounds built from non-unique constituent combinations (regardless 
# of the linker) appearing only in the Sg form and not undergoing the IntPE
# whose N1 mostly attach the pluralic linker in the Sg form
# (N=1560, UniqN=842, UniqCN=590, UniqN1N=37)
ic_alt_nipe_sg_l = pd.read_csv(
    os.path.join(sp18_ext_path, "ic_alt_nipe_sg_l.tsv"),
    sep="\t",
    header=0,
    index_col="gecodb",
    # all str, WC int, is linker pluralic bool
    dtype=defaultdict(
        lambda: str,
        wc="int",
        is_linker_pl="bool"
    ),
    quoting=3   # = csv.QUOTE_NONE, separate by tabs only
)

ic_alt_nipe_sg_l.sample(5)

,const_comb,n1,n2,n1_pl_marker,n1_sem,n2_num,is_linker_pl,wc,context,context_id
gecodb,,,,,,,,,,
stadt_+=e_tourist,stadt+tourist,stadt,tourist,_+=e_,ind,pl,True,634,Das traditionsreiche Haus wurde bereits 1913 a...,f9336e70-3ede-4061-9eeb-fa8916218084
ohr_+en_rand,ohr+rand,ohr,rand,_+en_,ind,pl,True,0,Auf eine besondere Schwierigkeit gilt es bei d...,8bf6ec80-d604-4672-ad2b-6e6954d9610f
hemd_+s_ärmel,hemd+ärmel,hemd,ärmel,_+en_,ind,pl,False,2199,aber dann hätten wir den Bandschleifer doch im...,cda59683-3005-4daa-a9aa-e369a9361202
nagel_+=_rand,nagel+rand,nagel,rand,_+=_,ind,pl,True,0,Das Ganze sah sehr unschön aus und war fast so...,5dafd67b-f2c5-4fb4-ba98-c04fc90ee443
nagel_+=_bombe,nagel+bombe,nagel,bombe,_+=_,ind,pl,True,0,"""""""Trotz der Waffenstillstandsvereinbarung , d...",27892baa-b198-4e74-bacb-5876f2b04f15


In [ ]:
# calculate absolute counts and shares of compounds 
# with pluralic linkers in Sg vs. Pl forms
ic_alt_nipe_sg_l_sum_data = {}
for const_comb, group in ic_alt_nipe_sg_l.groupby("const_comb"):
    for num in ["sg", "pl"]:
        ic_alt_nipe_sg_l_sum_data[(const_comb, f"num_{num}")] = {
            "l_sg": len(
                group[(group["n2_num"] == num) &
                (~group["is_linker_pl"])]
            ),
            "l_pl": len(
                group[(group["n2_num"] == num) &
                (group["is_linker_pl"])]
            )
        }

ic_alt_nipe_sg_l_sum = pd.DataFrame(ic_alt_nipe_sg_l_sum_data).T
ic_alt_nipe_sg_l_sum["pl_share"] = ic_alt_nipe_sg_l_sum["l_pl"] / (
    ic_alt_nipe_sg_l_sum["l_sg"] + ic_alt_nipe_sg_l_sum["l_pl"]
)
ic_alt_nipe_sg_l_sum.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sg_l/linker_counts.tsv"),
    **outcsv_args
)
# (UniqCN=590, UniqN1N=37),
# each constituent combination has 2 rows,
# one for Sg and one for Pl
ic_alt_nipe_sg_l_sum

l_sg  l_pl  pl_share
apfel+bauer      num_sg     1     0       0.0
                 num_pl     0     1       1.0
apfel+baum       num_sg     2     0       0.0
                 num_pl     2     3       0.6
apfel+dieb       num_sg     0     1       1.0
...                       ...   ...       ...
zitat+verwertung num_pl     1     0       0.0
zitat+zitat      num_sg     0     1       1.0
                 num_pl     1     0       0.0
zitat+zweck      num_sg     2     0       0.0
                 num_pl     1     0       0.0

[1180 rows x 3 columns]

In [ ]:
# retrieve pluralic linker shares only for Sg and Pl forms
ic_alt_nipe_sg_l_shares = ic_alt_nipe_sg_l_sum.unstack()["pl_share"]
ic_alt_nipe_sg_l_shares = ic_alt_nipe_sg_l_shares[["num_sg", "num_pl"]]
ic_alt_nipe_sg_l_shares.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sg_l/pl_l_shares.tsv"),
    **outcsv_args
)
# (UniqCN=590, UniqN1N=37)
ic_alt_nipe_sg_l_shares

,num_sg,num_pl
apfel+bauer,0.0,1.0
apfel+baum,0.0,0.6
apfel+dieb,1.0,1.0
apfel+hof,0.0,0.0
apfel+kopf,0.0,1.0
...,...,...
zitat+tag,0.0,0.0
zitat+text,1.0,1.0
zitat+verwertung,1.0,0.0
zitat+zitat,1.0,0.0


In [10]:
ic_alt_nipe_sg_l_shares_desc = ic_alt_nipe_sg_l_shares.describe()
ic_alt_nipe_sg_l_shares_desc.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sg_l/pl_l_desc.tsv"),
    **outcsv_args
)
ic_alt_nipe_sg_l_shares_desc

,num_sg,num_pl
count,590.000000,590.000000
mean,0.340847,0.394802
std,0.429162,0.457406
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,1.000000,1.000000
max,1.000000,1.000000


In [17]:
def sp18_orig_analysis(
    sp18_sample: pd.DataFrame
) -> Tuple[pd.Series, pd.Series, pd.Series]:

    # Methodology from the original [schaefer_pankratz_2018:26] paper:
    # 1. Create contingency table:
    # n2_num (Sg vs. Pl) x n1_pl_marker (zero vs. pluralic linker).
    # 2. Perform the chi-squared test of independence over the table
    # to assess the statistical significance of the association between
    # the Pl form of N2 and the presence of a pluralic linker in N1.
    # 3. Perform the Cramer's V test to assess the strength
    # of the association.
    # 4. Inspect symmetrical standardized residuals to identify
    # the cells that contribute to the association, and in which direction.

    # 1. Create contingency table replicating [schaefer_pankratz_2018:26]
    contingency_table = sp18_sample \
        .groupby(level=1)[["l_sg", "l_pl"]].sum()   \
        .T  \
        # [["num_sg", "num_pl"]]

    # 2. Perform the chi-squared test.

    # even though I didn't find the original R code
    # of [schaefer_pankratz_2018] for this experiment,
    # the `chi2_contingency` function from `scipy.stats`
    # produced the same results as in the paper
    # on the same contingency table, so we can be
    # confident that we are replicating the same analysis
    chi2_res = chi2_contingency(contingency_table)

    # 3. Perform the Cramer's V test to assess 
    # the strength of the association.

    # the original R code for Cramer's V is available in
    # linkingelements/Data/R/functions.R, and is as follows:
    # 
    #   cramer.v <- function(chisq) {
    #       unname(sqrt(chisq$statistic/(sum(chisq$observed)*min(dim(chisq$observed)-1))))
    #   }
    #
    # we will reconstruct it in Python:
    #   dim(chisq$observed) -> ic_alt_nipe_sg_l_ct.shape
    #
    #   min(dim(chisq$observed)-1) -> min(
    #       ic_alt_nipe_sg_l_ct.shape[0] - 1,
    #       ic_alt_nipe_sg_l_ct.shape[1] - 1
    #   )
    # 
    #   sum(chisq$observed) -> ic_alt_nipe_sg_l_ct.values.sum()
    #
    #   unname -> [irrelevant for Python]
    def cramer_v() -> float:
        return math.sqrt(
            chi2_res.statistic / (
                contingency_table.values.sum() *
                min(
                    contingency_table.shape[0] - 1,
                    contingency_table.shape[1] - 1
                )
            )
        )

    cramer_v_val = cramer_v()

    # the original R code for Cohen's W is available in
    # linkingelements/Data/R/functions.R, and is as follows:
    # 
    #   # cohen.w <- function(chisq) {
    #       cramer.v(chisq) * sqrt(min(dim(chisq$observed)-1))
    #   }
    def cohen_w() -> float:
        return cramer_v_val * math.sqrt(
            min(
                contingency_table.shape[0] - 1,
                contingency_table.shape[1] - 1
            )
        )

    cohen_w_val = cohen_w()

    outdf = pd.Series(
        data={
            "chi2": chi2_res.statistic.round(3),
            "df": chi2_res.dof,
            "p": chi2_res.pvalue,#.round(3),
            "cramer_v": round(cramer_v_val, 3),
            "cohen_w": round(cohen_w_val, 3)
        }
    )

    # 4. Inspect symmetrical standardized residuals to identify
    # the cells that contribute to the association, and in which direction.

    # even though I didn't find the original R code
    # of [schaefer_pankratz_2018] for this experiment,
    # the `standardized_resids` property of
    # `statsmodels.stats.contingency_tables.Table`
    # produced the same results as in the paper
    # on the same contingency table, so we can be
    # confident that we are replicating the same analysis
    ic_alt_nipe_sg_l_resids = Table(contingency_table)    \
        .standardized_resids.round(3)

    return contingency_table, outdf, ic_alt_nipe_sg_l_resids

In [18]:
ct, outdf, resids = sp18_orig_analysis(ic_alt_nipe_sg_l_sum)

ct.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sg_l/contingency_table.tsv"),
    **outcsv_args
)

outdf.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sg_l/chi2.tsv"),
    **{**outcsv_args, "header": False}
)

resids.to_csv(
    os.path.join(res_outdir, "ic_alt_nipe_sg_l/resids.tsv"),
    **outcsv_args
)

In [21]:
resids

,num_pl,num_sg
l_sg,-1.843,1.843
l_pl,1.843,-1.843
